# 02 · Five uplift models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/hillstrom-uplift-lab/blob/main/notebooks/02_uplift_models.ipynb)

**Goal:** estimate which customers may change their behavior because of one email campaign. We compare five methods on the same held-out randomized sample. The primary modeling outcome is purchase; change the configuration below to study visits instead.

## Run this notebook in Colab

Each notebook runs independently. The next cell installs the analysis packages, clones the project, and downloads the public dataset. You do not need Kaggle credentials.

In [ ]:
%pip -q install pandas numpy scipy statsmodels scikit-learn plotly


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/content/hillstrom-uplift-lab")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ericmavigo/hillstrom-uplift-lab.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "download_data.py"], check=True)
print(f"Project directory: {REPO_DIR}")


## 1. Choose the treatment and shared base learner

To compare uplift strategies fairly, all five use the same base learner and the same train/evaluation split. Logistic regression is a quick baseline; random forest can capture nonlinear patterns. The control group stays `No E-Mail`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from src.uplift import UPLIFT_MODELS, train_uplift_models, evaluate_uplift

df = pd.read_csv(REPO_DIR / "data" / "raw" / "hillstrom.csv")
TREATMENT = "Mens E-Mail"       # Change to "Womens E-Mail" to test the other campaign.
CONTROL = "No E-Mail"
OUTCOME = "conversion"          # Choose "visit" or "conversion".
BASE_LEARNER = "Logistic regression"  # Or "Random forest".
TEST_SHARE = 0.25

experiment = df[df.segment.isin([TREATMENT, CONTROL])].copy().reset_index(drop=True)
experiment["__treatment"] = (experiment.segment == TREATMENT).astype(int)
# Group exact rows together so identical observations cannot land in both partitions.
groups = pd.util.hash_pandas_object(experiment.drop(columns=["__treatment"]), index=False).astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SHARE, random_state=42)
train_idx, test_idx = next(splitter.split(experiment, groups=groups))
train = experiment.iloc[train_idx].copy()
test = experiment.iloc[test_idx].copy()
print(f"Training rows: {len(train):,}; held-out rows: {len(test):,}")
print(f"Treatment share: train={train.__treatment.mean():.3f}; test={test.__treatment.mean():.3f}")


## 2. Fit all five methods

The helper module is shared with the Streamlit executive dashboard so the notebook and app use the same implementation. All predictors are measured before email assignment. Treatment assignment is passed only where the method requires it; post-treatment outcomes are targets, never features.

In [ ]:
predictions = train_uplift_models(train, test, outcome=OUTCOME, family=BASE_LEARNER)
assert set(predictions) == set(UPLIFT_MODELS)
print("Scored evaluation customers with:", ", ".join(predictions))


## 3. Evaluate ranking value on held-out randomized customers

Qini compares incremental outcomes as we target customers in score order. The top-30% observed lift is the treatment-control outcome difference within the highest-scored group. Neither metric can observe both counterfactual outcomes for one person; both estimate ranking performance from randomized groups.

In [ ]:
scores, curves = evaluate_uplift(test, OUTCOME, predictions)
display(scores.style.format({"Qini area above random": "{:.5f}", "Top 30% observed lift": "{:+.2%}"}))


In [ ]:
import plotly.express as px

fig = px.line(curves, x="Targeted share", y="Incremental outcome gain", color="Model",
              title="Held-out Qini curves: uplift ranking versus random targeting")
fig.update_layout(xaxis_tickformat=".0%", yaxis_title="Estimated incremental outcome gain")
fig.show()


## 4. Interpret results and limits

- A higher Qini area means the model ranks customers with larger incremental outcomes earlier in the list.
- Compare with the random-targeting line; do not choose a model from training fit or accuracy alone.
- A negative or unstable held-out gain means there is not enough evidence that targeted ranking improves on random selection.
- The class-transformation derivation assumes close to balanced randomized assignment; the two selected arms in this experiment are close to 50/50.
- These models estimate heterogeneous effects but do not prove a profitable policy. Validate the selected policy in a new randomized campaign before deployment.

In [ ]:
winner = scores.iloc[0]
print(f"Highest held-out Qini score: {winner['Model']} (Qini={winner['Qini area above random']:.5f}).")
print("Treat this as a model-selection result for this split, not proof of future campaign profit.")
